<a href="https://colab.research.google.com/github/YKTTKY/Book-BuildALargeLanguageModel-FromScratch-/blob/main/BuildLLMFromScratch_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How We Tokenize Txt? (Byte Pair Encoding)?  
- use titoken pip package


In [ ]:
!pip install tiktoken

In [ ]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
#encode text example
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [ ]:
#decode integers example:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [ ]:
# Byte Pair Encoding of Unkown words :
# example 1:
uk_word_1 = "Gyatt"
ukw_1_integers = tokenizer.encode(uk_word_1)
d_ukw_1_int_2_str = tokenizer.decode(ukw_1_integers)
print(f"encode[Gyatt]:{ukw_1_integers}")
print(f"decode[Gyatt]: {d_ukw_1_int_2_str}")
ukw_1_subword_1 = ukw_1_integers[:len(ukw_1_integers) // 2]
ukw_1_subword_2 = ukw_1_integers[len(ukw_1_integers) // 2:]
# unknown words are tokenized into individual, characters or words.
print(f"sub encoded word 1: {tokenizer.decode(ukw_1_subword_1)}")
print(f"sub encoded word 2: {tokenizer.decode(ukw_1_subword_2)}")

encode[Gyatt]:[44802, 1078]
decode[Gyatt]: Gyatt
sub encoded word 1: Gy
sub encoded word 2: att


In the Essesnse, unknown words are converted to a list of tokens instead of one token.

## Data Sampling with a Sliding Window.
LLMs are pretrained by predicting the next word in a text.

<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/2-12.png"
alt="sliding window" >

Given a text sample, extract input blocks as subsamples that serve as input to the LLM, and the LLM’s prediction task during training is to predict the next word that follows the input block. During training, we mask out all words that are past the target. Note that the text shown in this figure must undergo tokenization before the LLM can process it; however, this figure omits the tokenization step for clarity.

In [ ]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path ="the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7a229c816390>)

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [ ]:
enc_sample = enc_text[50:]

In [ ]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print("input-target pairs for next-word prediction")
print(f"x: {x}")
print(f"y:      {y}")

input-target pairs for next-word prediction
x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [ ]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print("input:", context, "--predict next-->", " output: ",desired)

input: [290] --predict next-->  output:  4920
input: [290, 4920] --predict next-->  output:  2241
input: [290, 4920, 2241] --predict next-->  output:  287
input: [290, 4920, 2241, 287] --predict next-->  output:  257


In [ ]:
for i in range(1, context_size + 1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


## A Efficient Data Loader to handle iterations input and target pairs in multidimensional arrays - PyTorch Data Loader.

In a sliding window approach, these parameters determine how a long document is chopped into pieces that an LLM can actually digest. Since models have a fixed "memory" (context window), we use these settings to ensure no data is lost at the edges.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids  = []
    self.target_ids = []
    # tokenize the entire text
    token_ids = tokenizer.encode(txt)
    # use sliding window to chunk the book into overlapping sequence of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk  = token_ids[i   : i + max_length    ]
      target_chunk = token_ids[i+1 : i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))
  #Return the total number of rows in the dataset
  def __len__(self):
    return len(self.input_ids)
  #Return a single row from the dataset
  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size = 4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset   = GPTDatasetV1(txt, tokenizer, max_length, stride)
  dataloader = DataLoader(
      dataset,
      batch_size = batch_size,
      shuffle=shuffle,
      drop_last = drop_last,
      num_workers = num_workers
  )
  return dataloader

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/2-14.png" alt="stride batch window">

## Relationship between batch size, max length, and stride in sliding window tokenization.  

1. **Max Length (The Container Size)**

    **Max Length** is the total number of tokens in a single "chunk" or "window."  

    - It is usually dictated by the model's limit (e.g., 512, 1024, or 32k tokens).

    - If a document is 2,000 tokens and your Max Length is 512, the document must be split into multiple segments.
2. **Stride (The "Step" Size)**  

    **Stride** is the number of tokens the window moves forward to start the next chunk. This is the most critical part of the sliding window logic.
    - **Overlap**: If the Stride is smaller than the Max Length, you create an overlap.
    - **The "Context Gap"** Fix: If you split text exactly at the Max Length with no overlap, the model might lose the relationship between the last word of Chunk 1 and the first word of Chunk 2. Overlap ensures the model sees the "bridge" between segments.  
    **Calculation: Overlap=Max Length−Stride**
3. **Batch Size (The Parallel Processor)**  

    While Max Length and Stride define **how** the text is cut, **Batch Size** defines **how many** of those chunks are processed by the GPU at the exact same time.
    - If your document results in 10 chunks and your Batch Size is 4, the GPU will process them in groups of 4, 4, and then the remaining 2.
    - Higher batch sizes speed up training/inference but require more VRAM.
---
| Parameter | Function | Impact |
| -------- | -------- | -------- |
| Max Length | Window Width | Defines the "field of view" for the model. |
| Stride | Movement | Control how much information is repeated between windows. |
| Batch Size | Parallelism| Determines memory usage and processing speed. |
---
#### A Concrete Example:
Imagine a document with 100 tokens.
- max length = 40
- stride = 30
| window | token range | calculation |
| --------| -------- | --------|
| window1| 1 to 40     | start at 1, takes 40 tokens|
| window2| 31 to 70    | start at 31, takes 70 tokens|
|window3 | 61 to 100 | start at 61 takes 40 tokens|


In [ ]:
#different batch size:
data_loader_b = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter_b = iter(data_loader_b)
inputs, targets = next(data_iter_b)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating token embeddings
The Last Step in preparing the input text for LLM training is to convert the token IDs into embedding vectors.
<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/2-15.png" alt="workflow of pretraining.">

In [ ]:
# demo: how token IDs to embedding vectors:
demo_input_ids = torch.tensor([2,3,5,1])
# parameters to instantiate embedding layer in pytorch
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


The weight matrix of the embedding layer contains small, random values. These values are optimized during LLM training as part of the LLM optimization itself. Moreover, we can see that the weight matrix has six rows and three columns. There is one row for each of the six possible tokens in the vocabulary, and there is one column for each of the three embedding dimensions.

In [ ]:
# let's apply it to a token ID to obtain the embedding vector:
print(embedding_layer(torch.tensor([3])))
# exactly identical to the forth rows.

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/2-16.png" alt="token ids to embedding">

## Encoding Word Position.  
LLM self-attention mechanism itself doesn't have a notion of position or order for the tokens within a sequence.  

What exactly means is the model sees words as a "set" or a "cloud" because the self-attention mechanism processes all words at once in parallel, it doesn't naturally know which word came first. To the model, these two sentences look identical:  

- "The dog bit the man."
- "The man bit the dog."  

The meaning is completely different, but the "bag of vectors" is the same. This is what "position-agnostic" really means.  

1. **The "Deterministic" Problem**   

    Embedding a token ID is "deterministic" and "position-independent".
    - **what this means**: The word "Apple" will always result in the same vector(e.g., `[0.1, -0.5, 0.9]`), whether it is the first word in a book or the last.
    - **The Issue**: If the embedding doesn't change based on where it is, and the attention mechanism treats the sequnce like a pile of data rather than a timeline, the model losses the flow of "language".

2. **Why Self-Attention is Position Agnostic**  

    In older models (like RNNs or LSTMs), the model processed words one by one, like a human reading. It had a "memory" that naturally updated as it moved from left to right.  
    Transformers threw that away for speed. They look at the entire sequence at once. If you swapped the first and last words of a 1,000-word essay, the raw self-attention scores would essentially be the same because it’s just calculating the relationship between pairs of words, not their place in the "line."

3. **The Solution: Positional Embedding**  

    To fix this, we "inject" position information. We take the standard word vector and add a "position vector" to it.  
    ``` Final Input=Token Embedding + Positional Encoding ```  
    Imagine the word "Bank":
      - At **Position 1**, we add a specific "Position 1" signature.
      - At **Position 50**, we add a specific "Position 50" signature.  

    Now, even though the word "Bank" is the same, the Final Input vector is slightly different at the start of the sentence than it is at the end. This allows the self-attention mechanism to realize, "Ah, this word is near the beginning, and that word is near the end," allowing it to understand syntax and grammar.




In [ ]:
# absolute position embedding:
# create a 256-dimensional vector representation:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
# instantiate dataloader:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
# from token ids to 256-dimensional vectors:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
# create another embedding layer for positional embeddings.
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [ ]:
# add these embeddings directly: add 4x256-dimensional pos_embedding tensor to each 4x256-dimensional token embedding tensor in each of the eight batches.
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


<img src="https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781633437166/files/Images/2-19.png" alt="input processing pipeline">

## Summary
- LLMs require textual data to be converted into numerical vectors, known as embeddings, since they can’t process raw text. Embeddings transform discrete data (like words or images) into continuous vector spaces, making them compatible with neural network operations

- As the first step, raw text is broken into tokens, which can be words or characters. Then, the tokens are converted into integer representations, termed token IDs.

- Special tokens, such as `<|unk|>` and `<|endoftext|>`, can be added to enhance the model’s understanding and handle various contexts, such as unknown words or marking the boundary between unrelated texts.

- The byte pair encoding (BPE) tokenizer used for LLMs like GPT-2 and GPT-3 can efficiently handle unknown words by breaking them down into subword units or individual characters.

- We use a sliding window approach on tokenized data to generate input–target pairs for LLM training.

- Embedding layers in PyTorch function as a lookup operation, retrieving vectors corresponding to token IDs. The resulting embedding vectors provide continuous representations of tokens, which is crucial for training deep learning models like LLMs.

- While token embeddings provide consistent vector representations for each token, they lack a sense of the token’s position in a sequence. To rectify this, two main types of positional embeddings exist: absolute and relative. OpenAI’s GPT models utilize absolute positional embeddings, which are added to the token embedding vectors and are optimized during the model training.